In [ ]:
import pandas as pd

data = pd.read_csv('Social Media Engagement Dataset.csv')
data.head()

In [ ]:

cleaned_data = data.drop(columns=['post_id', 'user_id'])

cleaned_data.columns

In [ ]:
cleaned_data.head()

In [ ]:
list(cleaned_data.columns)

In [ ]:
from sklearn.model_selection import train_test_split

X_raw = encoded_data.drop(columns=['engagement_rate'])
y = encoded_data['engagement_rate']

X = X_raw.select_dtypes(include=['number'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Cleaned training clues shape: {X_train.shape}")
print(f"Cleaned training clues shape: {X_test.shape}")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models


model = models.Sequential()

model.add(layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)))

model.add(layers.Dense(32, activation='relu'))

model.add(layers.Dense(1))

model.summary()

In [ ]:
X_train.dtypes

In [ ]:
model.compile(
    optimizer='adam', 
    loss='mean_squared_error', 
    metrics=['mae']
)

print("Factory compiled! Starting the 20 training rounds now!\n")

history = model.fit(
    X_train, 
    y_train, 
    validation_data=(X_test, y_test), 
    epochs=20, 
    batch_size=32
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(history.history['mae'], label='Training Error (MAE)')
plt.plot(history.history['val_mae'], label='Validation Error (MAE)')
plt.title('AI Learning Curve')
plt.xlabel('Epochs')
plt.ylabel('Error Rate')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split


viral_threshold = encoded_data['engagement_rate'].quantile(0.90)

y_viral = (encoded_data['engagement_rate'] > viral_threshold).astype(int)

X_raw_triggers = encoded_data.drop(columns=[
    'engagement_rate', 'likes_count', 'shares_count', 'comments_count', 'impressions'
])

X_triggers = X_raw_triggers.select_dtypes(include=['number'])

X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(
    X_triggers, y_viral, test_size=0.2, random_state=42
)

print(f"Trigger columns left to predict with: {list(X_triggers.columns)}")
print(f"Viral posts in training data: {sum(y_train_v)} out of {len(y_train_v)}")

In [ ]:
from tensorflow.keras import layers, models

viral_model = models.Sequential()


viral_model.add(layers.Dense(64, activation='relu', input_shape=(X_train_v.shape[1],)))
viral_model.add(layers.Dense(32, activation='relu'))

viral_model.add(layers.Dense(1, activation='sigmoid'))

viral_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

viral_model.summary()

In [ ]:
print("Starting the Virality Classifier Training Pipeline...\n")

history_viral = viral_model.fit(
    X_train_v, 
    y_train_v, 
    validation_data=(X_test_v, y_test_v), 
    epochs=20, 
    batch_size=32
)

In [ ]:
import numpy as np

total_posts = len(y_train_v)
num_non_viral = total_posts - sum(y_train_v)
num_viral = sum(y_train_v)

class_weight = {
    0: 1.0,
    1: float(num_non_viral / num_viral)
}

print(f"Mathematical punishment weight for missing a viral post: {class_weight[1]:.2f}x\n")

viral_model_fixed = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_v.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

viral_model_fixed.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_fixed = viral_model_fixed.fit(
    X_train_v, 
    y_train_v, 
    validation_data=(X_test_v, y_test_v), 
    epochs=20, 
    batch_size=32,
    class_weight=class_weight
)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint
import matplotlib.pyplot as plt


checkpoint_callback = ModelCheckpoint(
    filepath='best_viral_model.keras',
    monitor='val_accuracy',
    mode='max',
    save_best_only=True,
    verbose=1
)

final_viral_model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_v.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

final_viral_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Training with the Automatic Maximum Tracker Activated...\n")
history_final = final_viral_model.fit(
    X_train_v, 
    y_train_v, 
    validation_data=(X_test_v, y_test_v), 
    epochs=20, 
    batch_size=32,
    class_weight=class_weight,
    callbacks=[checkpoint_callback]
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history_final.history['val_accuracy'], label='Validation Accuracy', color='purple', marker='o')
plt.title('Finding the Peak Virality Accuracy Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy Score')
plt.legend()
plt.grid(True)
plt.show()